# LLM Training Pipeline: From Raw Data to Aligned Model

> Complete flow from data collection through Pre-training → SFT → RLHF → PPO, with labelled relations at every step.

***

## Pipeline Overview

```
┌─────────────────────────────────────────────────────────────────────┐
│                    LLM TRAINING PIPELINE                            │
│         Raw Data  →  Base Model  →  SFT  →  RLHF  →  Deployed       │
└─────────────────────────────────────────────────────────────────────┘
```

***

## Stage 0 — Data Collection & Preparation

**Files:** `pretrain.py`, `tokenizer.json`

```
┌─────────────────────────────────────────────────────┐
│                  DATA SOURCES                       │
│                                                     │
│  Web Crawl      Books &       Code Repos    Papers  │
│  (Common        Wikipedia     (GitHub)      (arXiv) │
│  Crawl, C4)                                         │
└──────────────────────┬──────────────────────────────┘
                       │
                       │  collect & merge raw text
                       ▼
┌─────────────────────────────────────────────────────┐
│                PROCESSING STEPS                     │
│                                                     │
│  1. Deduplication & quality filtering               │
│  2. Tokenisation → build stoi / itos vocab maps     │
│  3. encode(text) → integer token IDs                │
│  4. Split: train / val / test                       │
└──────────────────────┬──────────────────────────────┘
                       │
                       │  outputs
                       ▼
        ┌──────────────────────────────┐
        │  tokenizer.json              │  ← stoi + itos vocab maps
        │  train_data.bin              │  ← encoded token IDs
        │  vocab_size = N              │  ← used to build model
        └──────────────────────────────┘
```

***

## Stage 0 → Stage 1 Relation

```
tokenizer.json  ──[encode() feeds token IDs]──►  Pre-training loop
vocab_size      ──[sets embedding table size]──►  nn.Embedding(vocab_size, vocab_size)
```

***

## Stage 1 — Pre-Training (Base Model)

**Files:** `pretrain.py` → `trained-model/bigram_pretrained.pt`, `config.json`

```
┌─────────────────────────────────────────────────────┐
│               PRE-TRAINING LOOP                     │
│                                                     │
│  Objective : next-token prediction                  │
│  Loss      : F.cross_entropy(logits, targets)       │
│  Optimizer : AdamW   LR = 3e-4  (aggressive)        │
│  Steps     : 5,000+  on massive corpus              │
│  Block size: 256 tokens                             │
│                                                     │
│  for step in range(max_steps):                      │
│      x, y  = get_batch("train")                     │
│      logits, loss = model(x, y)   ← forward         │
│      loss.backward()              ← backward        │
│      optimizer.step()             ← update weights  │
└──────────────────────┬──────────────────────────────┘
                       │
                       │  outputs
                       ▼
        ┌──────────────────────────────┐
        │  bigram_pretrained.pt        │  ← learned weights
        │  config.json                 │  ← model_type, vocab_size
        └──────────────────────────────┘

  Result: Base Model
  ✔ Knows language, facts, reasoning
  ✘ Does NOT follow instructions yet
```

***

## Stage 1 → Stage 2 Relation

```
bigram_pretrained.pt ──[load_state_dict()]──► SFT model starting point
config.json          ──[vocab_size]─────────► BigramLanguageModel(vocab_size)
tokenizer.json       ──[shared vocab]───────► same encode/decode in SFT
```

> **Key principle:** SFT starts from pre-trained weights — not random init.
> This is identical to `AutoModel.from_pretrained(...)` in HuggingFace.

***

## Stage 2 — Supervised Fine-Tuning (SFT)

**Files:** `finetune.py` → `sft-model/bigram_sft.pt`

```
┌─────────────────────────────────────────────────────┐
│              SFT DATASET FORMAT                     │
│              (Alpaca / LLaMA-chat style)            │
│                                                     │
│  ### Instruction:                                   │
│  What is gravity?                                   │
│                                                     │
│  ### Response:                                      │
│  Gravity is a force that pulls objects...           │
│  ──────────────────────────────────────             │
│  × 200 repetitions  (or ~52k unique pairs           │
│  in real LLMs like InstructGPT)                     │
└──────────────────────┬──────────────────────────────┘
                       │
                       │  same cross-entropy loss, different data
                       ▼
┌─────────────────────────────────────────────────────┐
│               SFT TRAINING LOOP                     │
│                                                     │
│  LR      = 1e-4       ← lower than pre-training     │
│  Steps   = 2,000      ← fewer, stop before overfit  │
│  Val split monitored  ← new vs pre-training         │
│  Loss on response tokens only                       │
│                                                     │
│  Key difference from pre-training:                  │
│  SAME loss function, DIFFERENT data quality         │
└──────────────────────┬──────────────────────────────┘
                       │
                       │  outputs
                       ▼
        ┌──────────────────────────────┐
        │  sft-model/bigram_sft.pt     │  ← instruction-following weights
        └──────────────────────────────┘

  Result: Instruction Model
  ✔ Follows prompts & answers questions
  ✘ May still be harmful or unhelpful
```

***

## Stage 2 → Stage 3 Relation

```
bigram_sft.pt  ──[load_state_dict()]────────────► m  (active policy, will be updated)
bigram_sft.pt  ──[copy.deepcopy() + freeze]─────► m_ref (KL anchor, never updated)
reward_model   ──[trained in 3b, then frozen]───► scores m's responses in PPO loop
```

> **Why load from SFT, not pre-trained?**
> RLHF needs a model that already follows instructions.
> Running PPO on a raw base model gives the reward signal nothing stable to work with.

***

## Stage 3 — RLHF (Reinforcement Learning from Human Feedback)

### Stage 3a — Generate Candidate Responses

```
┌─────────────────────────────────────────────────────┐
│           RESPONSE GENERATION                       │
│                                                     │
│  For each prompt:                                   │
│    generate(temp=0.5)  → conservative response A    │
│    generate(temp=1.0)  → balanced response B        │
│    generate(temp=1.5)  → creative response C        │
└──────────────────────┬──────────────────────────────┘
                       │
                       │  responses shown to humans
                       ▼
┌─────────────────────────────────────────────────────┐
│           HUMAN LABELLING                           │
│                                                     │
│  Crowdworkers rank: A > B > C                       │
│  Criteria: Helpful · Harmless · Honest (3H)         │
│  InstructGPT scale: ~12,000 comparisons             │
└──────────────────────┬──────────────────────────────┘
                       │
                       │  outputs
                       ▼
        ┌──────────────────────────────┐
        │  preference pairs (A > B)    │  ← training signal for RM
        └──────────────────────────────┘
```

***

### Stage 3b — Train the Reward Model (RM)

```
┌─────────────────────────────────────────────────────┐
│           REWARD MODEL ARCHITECTURE                 │
│                                                     │
│  Base    : copy of SFT model                        │
│  Head    : LM head replaced → scalar output head    │
│  Output  : single score per response (0.0 – 1.0)   │
│  Loss    : MSELoss on (predicted_score, human_rank) │
└──────────────────────┬──────────────────────────────┘
                       │
                       │  trained on preference pairs
                       ▼
        ┌──────────────────────────────┐
        │  reward_model.pt  (FROZEN)   │  ← never updated after training
        └──────────────────────────────┘

  high score  →  humans prefer this response
  low score   →  humans dislike this response
  acts as proxy for human judgement in the PPO loop
```

***

### Stage 3c — PPO Fine-Tuning

```
┌─────────────────────────────────────────────────────┐
│           THREE ACTORS IN PPO                       │
│                                                     │
│  m        (Policy)      ← UPDATED each step         │
│  rm       (Reward Model)← FROZEN judge              │
│  m_ref    (Reference)   ← FROZEN SFT anchor         │
└─────────────────────────────────────────────────────┘

PPO Loss:
  rl_loss = (1 - reward) × lm_loss  +  β × KL(m ∥ m_ref)
                ↑                            ↑
         high reward = small update    prevents reward hacking

LR = 1e-5  ← smallest learning rate of all three stages
```

**The PPO Inner Loop (repeated every step):**

```
  ┌──────────┐     sample response      ┌──────────┐
  │          │ ─────────────────────►   │          │
  │  Policy  │                          │  Reward  │
  │   (m)    │ ◄─────────────────────   │  Model   │
  │          │     scalar score         │   (rm)   │
  └────┬─────┘                          └──────────┘
       │
       │  compare to m_ref
       ▼
  ┌──────────┐
  │  m_ref   │  KL penalty = how far has policy drifted?
  │ (frozen) │  large drift → penalise → stay close to SFT
  └────┬─────┘
       │
       │  compute rl_loss
       ▼
  ┌──────────────────────┐
  │  Backprop + Update   │  nudge m weights toward higher reward
  │  optimizer.step()    │  while staying near m_ref
  └──────────────────────┘
```

**Why PPO instead of standard gradient descent?**

```
  next_id = torch.multinomial(probs, 1)
                  ↑
         SAMPLING — not differentiable!
         Cannot compute d(reward)/d(weights) through a random sample.
         PPO handles non-differentiable actions via RL policy gradients.
```

***

## Stage 3 → Deployment Relation

```
bigram_rlhf.pt   ──[ships to users]──────► inference.py loads this
reward_model.pt  ──[stays internal]──────► never shipped (OpenAI never released theirs either)
m_ref            ──[discarded after PPO]─► training artifact only
```

***

## Full End-to-End Graph

```
[Raw Internet Text]
[Books / Wikipedia]    ──collect──►  [Data Processing]  ──tokenise──►  tokenizer.json
[Code / Papers]                        dedup, filter                    train_data.bin
                                                                        vocab_size
                                              │
                                    encode() feeds IDs
                                              │
                                              ▼
                                    [Stage 1: Pre-Training]
                                     next-token prediction
                                     LR=3e-4, 5000+ steps
                                     cross_entropy loss
                                              │
                                    bigram_pretrained.pt
                                    config.json
                                              │
                                  load_state_dict() from pretrained
                                              │
                                              ▼
                                    [Stage 2: SFT]
                                     instruction → response pairs
                                     LR=1e-4, 2000 steps
                                     same loss, curated data
                                              │
                                    sft-model/bigram_sft.pt
                                              │
                               ┌──────────────┴──────────────┐
                          load_state_dict()            copy.deepcopy() + freeze
                               │                             │
                               ▼                             ▼
                            m (policy)                   m_ref (anchor)
                          will be updated               never updated
                               │
                    ┌──────────┴───────────────────────────────────┐
                    │           Stage 3: RLHF                      │
                    │                                              │
                    │  3a. SFT generates N responses per prompt    │
                    │      humans rank them  A > B > C             │
                    │                  │                           │
                    │      preference pairs                        │
                    │                  │                           │
                    │  3b. Train Reward Model on rankings          │
                    │      RM: SFT backbone + scalar head          │
                    │      MSELoss → scores responses              │
                    │      FREEZE reward_model.pt                  │
                    │                  │                           │
                    │      scalar score (per response)             │
                    │                  │                           │
                    │  3c. PPO Loop (per step):                    │
                    │      ① m generates response                  │
                    │      ② rm scores it  → reward               │
                    │      ③ KL(m ∥ m_ref) → drift penalty        │
                    │      ④ rl_loss = (1-reward)×lm_loss          │
                    │               + β × KL_penalty               │
                    │      ⑤ backprop + optimizer.step()           │
                    │         LR=1e-5, nudge m weights             │
                    └──────────────────────┬───────────────────────┘
                                           │
                                 rlhf-model/bigram_rlhf.pt
                                           │
                                           ▼
                              ┌─────────────────────────┐
                              │   DEPLOYED MODEL        │
                              │                         │
                              │  Ships:                 │
                              │  • tokenizer.json       │
                              │  • config.json          │
                              │  • model.py             │
                              │  • bigram_rlhf.pt       │
                              │                         │
                              │  Does NOT ship:         │
                              │  • reward_model.pt      │
                              │  • m_ref                │
                              │  • training data        │
                              └─────────────────────────┘
```

***

## File Inventory by Stage

| Stage | Input Files | Output Files | Notes |
|-------|-------------|--------------|-------|
| 0 — Data Prep | raw text | `tokenizer.json`, `train_data.bin` | Same tokenizer used in all stages |
| 1 — Pre-training | `tokenizer.json`, `train_data.bin` | `bigram_pretrained.pt`, `config.json` | Random init → trained weights |
| 2 — SFT | `bigram_pretrained.pt`, `config.json` | `sft-model/bigram_sft.pt` | Starts from pretrained, not random |
| 3a — Generate | `bigram_sft.pt` | `preference_pairs.json` | Human rankings collected here |
| 3b — Reward Model | `preference_pairs.json` | `reward_model.pt` | Frozen after training |
| 3c — PPO | `bigram_sft.pt`, `reward_model.pt` | `rlhf-model/bigram_rlhf.pt` | m_ref is internal, discarded |
| Deploy | `bigram_rlhf.pt`, `config.json`, `tokenizer.json`, `model.py` | — | reward_model.pt NOT shipped |

***

## Key Concepts at Each Arrow

| Relation | What Transfers | Why It Matters |
|----------|---------------|----------------|
| Stage 0 → 1 | Token IDs via `encode()` | Model never sees raw text — only integers |
| Stage 1 → 2 | `load_state_dict(pretrained.pt)` | SFT refines, not retrains from scratch |
| Stage 2 → 3 | `load_state_dict(sft.pt)` + `deepcopy` freeze | PPO needs an already-instruction-following model |
| 3a → 3b | Human preference rankings | Only place human signal enters the pipeline |
| 3b → 3c | Scalar reward score | RM is a proxy — humans not in the loop during PPO |
| KL penalty | `KL(m ∥ m_ref)` | Prevents model drifting into reward-hacking nonsense |
| 3c → Deploy | `bigram_rlhf.pt` | Aligned weights ship; reward model stays private |